# HLO 500 Scenario Response Evaluation
Read scenarios and raw before/after recommendation responses; calculate every trial metric using `src/hlo/evaluation/metrics.py`; export two Excel workbooks. Raw ranked lists and relevance sets are required. This notebook does not generate metric values from Table 5 targets. It is a new execution harness, not a recovered historical runner.

## 1. Dependencies

In [ ]:
import sys
import importlib.util
required = {"pandas": "pandas", "numpy": "numpy", "openpyxl": "openpyxl", "rdflib": "rdflib"}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    raise ImportError("Install dependencies with: " + sys.executable + " -m pip install " + " ".join(missing))

## 2. Project and input configuration

In [ ]:
from pathlib import Path
import hashlib, json, platform
from collections import Counter
import numpy as np
import pandas as pd
from rdflib import Graph, RDF, RDFS, OWL, URIRef
from IPython.display import display

PROJECT_OVERRIDE = None
SCENARIO_OVERRIDE = None
RESPONSE_BEFORE_OVERRIDE = None
RESPONSE_AFTER_OVERRIDE = None
RELEVANCE_OVERRIDE = None
OUTPUT_OVERRIDE = None
TOP_K = 5

# Override the paths above when the actual filenames differ.
def locate_project():
    if PROJECT_OVERRIDE is not None:
        return Path(PROJECT_OVERRIDE).resolve()
    for parent in (Path.cwd(), *Path.cwd().parents):
        for candidate in (parent, parent / "HLO-main"):
            if (candidate / "src/hlo/evaluation/metrics.py").is_file() and (candidate / "evaluation/data/scenarios.csv").is_file():
                return candidate
    raise FileNotFoundError("Set PROJECT_OVERRIDE to HLO-main")

PROJECT = locate_project()
EVALUATION = PROJECT / "evaluation"
SCENARIOS_PATH = Path(SCENARIO_OVERRIDE) if SCENARIO_OVERRIDE else EVALUATION / "data/scenarios.csv"
RESPONSES = {
    "before": Path(RESPONSE_BEFORE_OVERRIDE) if RESPONSE_BEFORE_OVERRIDE else EVALUATION / "data/responses_before.jsonl",
    "after": Path(RESPONSE_AFTER_OVERRIDE) if RESPONSE_AFTER_OVERRIDE else EVALUATION / "data/responses_after.jsonl",
}
ONTOLOGIES = {"before": EVALUATION / "ontologies/before/HLO_T1.owl", "after": EVALUATION / "ontologies/after/HLO_Template_LLM.owl"}
RELEVANCE_PATH = Path(RELEVANCE_OVERRIDE) if RELEVANCE_OVERRIDE else None
OUTPUT = Path(OUTPUT_OVERRIDE) if OUTPUT_OVERRIDE else EVALUATION / "response_results"
OUTPUT.mkdir(parents=True, exist_ok=True)
METRICS = ["top1", "hit3", "hit5", "mrr", "map"]
sys.path.insert(0, str(PROJECT / "src"))
from hlo.evaluation.metrics import hit_at_k, reciprocal_rank, average_precision

display(pd.DataFrame([{"role": "scenarios", "path": str(SCENARIOS_PATH)},
    *[{"role": release + " responses", "path": str(path)} for release, path in RESPONSES.items()],
    {"role": "output", "path": str(OUTPUT)}]))

## 3. Required response schema

In [ ]:
response_contract = {
    "format": "JSONL, JSON array, JSON object with responses array, or CSV with JSON list columns",
    "required": ["scenario_id", "trial", "ranked_resource_ids"],
    "relevance": "relevant_resource_ids in each response, or an external CSV with scenario_id and relevant_resource_ids",
    "trial_values": [1, 2],
    "records_per_release": 1000,
    "scenario_count": 500,
    "optional_provenance": ["seed", "seed_role", "deterministic", "ontology_sha256", "resources_sha256", "runner_sha256", "top_k"],
    "seed_policy": "record supplied seeds; do not invent or infer trial seeds from selection seeds",
}
display(pd.DataFrame(response_contract.items(), columns=["item", "requirement"]))

## 4. Load and verify scenario identifiers

In [ ]:
if not SCENARIOS_PATH.is_file():
    raise FileNotFoundError(SCENARIOS_PATH)
scenarios = pd.read_csv(SCENARIOS_PATH, keep_default_na=False)
if "scenario_id" not in scenarios:
    raise ValueError("Scenario CSV must contain explicit scenario_id values shared with responses")
scenarios["scenario_id"] = scenarios.scenario_id.astype(str)
if len(scenarios) != 500 or not scenarios.scenario_id.is_unique or scenarios.scenario_id.eq("").any():
    raise ValueError("Expected exactly 500 unique, nonempty scenario IDs")
SCENARIO_IDS = set(scenarios.scenario_id)
if "scenario_type" in scenarios:
    display(scenarios.scenario_type.value_counts().rename("scenarios").reset_index())
display(scenarios.head(10))

## 5. Scenario sources and ontology roots

In [ ]:
def local(value):
    return str(value).rsplit("#", 1)[-1].rsplit("/", 1)[-1]

def ontology_index(path):
    graph = Graph().parse(path)
    source_pred = next((p for p in set(graph.predicates()) if local(p) == "sourceIdentifier"), None)
    index = {}
    if source_pred:
        for subject in graph.subjects(RDF.type, OWL.Class):
            for sid in graph.objects(subject, source_pred):
                index[str(sid)] = subject
    return graph, index

def root_paths(graph, subject):
    pending, paths = [(subject, [str(subject)])], []
    while pending:
        current, route = pending.pop()
        parents = sorted(graph.objects(current, RDFS.subClassOf), key=str)
        if not parents:
            paths.append([local(node) for node in route])
        for parent in parents:
            if str(parent) in route:
                paths.append([local(node) for node in route] + ["cycle"])
            else:
                pending.append((parent, route + [str(parent)]))
    return paths

source_rows = []
for release, path in ONTOLOGIES.items():
    if not path.is_file():
        continue
    graph, index = ontology_index(path)
    for scenario in scenarios.to_dict("records"):
        sid = scenario.get("source_identifier", "")
        subject = index.get(str(sid))
        source_rows.append({"release": release, "scenario_id": scenario["scenario_id"],
            "source_identifier": sid, "resolved_in_owl": subject is not None,
            "class_iri": str(subject) if subject is not None else "",
            "root_paths": json.dumps(root_paths(graph, subject)) if subject is not None else "[]"})
source_resolution = pd.DataFrame(source_rows)
if not source_resolution.empty:
    source_resolution.to_csv(OUTPUT / "scenario_ontology_roots.csv", index=False)
    display(source_resolution.head(8))
print("Source resolution is recorded separately; it does not create resource responses.")

## 6. Response readers

In [ ]:
def read_responses(path):
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path, keep_default_na=False).to_dict("records")
    if suffix == ".jsonl":
        return [json.loads(line) for line in path.read_text(encoding="utf-8-sig").splitlines() if line.strip()]
    document = json.loads(path.read_text(encoding="utf-8-sig"))
    if isinstance(document, dict):
        document = document.get("responses")
    if not isinstance(document, list):
        raise ValueError(f"{path}: expected a response array")
    return document

def id_list(value, field):
    if isinstance(value, str):
        value = json.loads(value)
    if not isinstance(value, list):
        raise ValueError(f"{field} must be a list or JSON list")
    result = []
    for item in value:
        if not isinstance(item, (str, int)) or isinstance(item, bool) or str(item) == "":
            raise ValueError(f"Invalid resource ID in {field}: {item!r}")
        result.append(str(item))
    if len(result) != len(set(result)):
        raise ValueError(f"Duplicate IDs in {field}")
    return result

missing_responses = [str(path) for path in RESPONSES.values() if not path.is_file()]
if missing_responses:
    raise FileNotFoundError("Original response files are missing. Provide raw rankings, not Excel metric columns: " + "; ".join(missing_responses))
raw = {release: read_responses(path) for release, path in RESPONSES.items()}
print({release: len(rows) for release, rows in raw.items()})

## 7. Optional external relevance judgments

In [ ]:
external_gold = {}
if RELEVANCE_PATH is not None:
    gold = pd.read_csv(RELEVANCE_PATH, keep_default_na=False)
    if not {"scenario_id", "relevant_resource_ids"}.issubset(gold.columns):
        raise ValueError("Gold CSV requires scenario_id and relevant_resource_ids")
    if not gold.scenario_id.astype(str).is_unique:
        raise ValueError("Duplicate scenario ID in gold file")
    external_gold = {str(r["scenario_id"]): set(id_list(r["relevant_resource_ids"], "relevant_resource_ids")) for r in gold.to_dict("records")}
    if set(external_gold) != SCENARIO_IDS:
        raise ValueError("Gold IDs must exactly match the 500 scenarios")
print("External gold sets:", len(external_gold))

## 8. Normalize responses and preserve seed provenance

In [ ]:
normalized = {}
provenance_rows = []
for release, rows in raw.items():
    cleaned = []
    for number, row in enumerate(rows, 1):
        if not {"scenario_id", "trial", "ranked_resource_ids"}.issubset(row):
            raise ValueError(f"{release} row {number}: missing scenario_id, trial or raw ranked_resource_ids")
        if str(row["trial"]) not in {"1", "2", "1.0", "2.0"}:
            raise ValueError(f"Invalid trial: {row['trial']}")
        sid, trial = str(row["scenario_id"]), int(float(row["trial"]))
        if sid not in SCENARIO_IDS:
            raise ValueError(f"Unknown scenario ID: {sid}")
        if row.get("release", release) != release:
            raise ValueError("Response release does not agree with input filename")
        ranked = id_list(row["ranked_resource_ids"], "ranked_resource_ids")
        supplied_gold = set(id_list(row["relevant_resource_ids"], "relevant_resource_ids")) if "relevant_resource_ids" in row else None
        relevant = external_gold.get(sid, supplied_gold)
        if relevant is None:
            raise ValueError(f"Missing relevance judgments: {release}/{sid}/{trial}")
        if supplied_gold is not None and sid in external_gold and supplied_gold != external_gold[sid]:
            raise ValueError("Response and external relevance judgments disagree")
        seed = row.get("seed")
        if seed == "":
            seed = None
        if seed is not None:
            if isinstance(seed, bool) or (isinstance(seed, float) and not seed.is_integer()):
                raise ValueError("Seed must be an integer")
            seed = int(seed)
        if row.get("top_k") is not None and int(row["top_k"]) < TOP_K:
            raise ValueError("Response truncation is smaller than evaluation TOP_K")
        cleaned.append({"release": release, "scenario_id": sid, "trial": trial,
                        "ranked_resource_ids": ranked[:TOP_K], "relevant_resource_ids": sorted(relevant), "seed": seed})
        provenance_rows.append({"release": release, "scenario_id": sid, "trial": trial,
            "seed": seed, "seed_role": row.get("seed_role", "unspecified"),
            "deterministic": row.get("deterministic", "unspecified"),
            "ontology_sha256": row.get("ontology_sha256", ""),
            "resources_sha256": row.get("resources_sha256", ""),
            "runner_sha256": row.get("runner_sha256", "")})
    normalized[release] = cleaned
provenance = pd.DataFrame(provenance_rows)
provenance.to_csv(OUTPUT / "response_trial_provenance.csv", index=False)
display(provenance.head(8))

## 9. Verify complete 500 by 2 response coverage

In [ ]:
expected_keys = {(sid, trial) for sid in SCENARIO_IDS for trial in (1, 2)}
for release, rows in normalized.items():
    keys = [(r["scenario_id"], r["trial"]) for r in rows]
    if len(keys) != 1000 or len(keys) != len(set(keys)) or set(keys) != expected_keys:
        raise ValueError(f"{release}: require exactly one raw response for every scenario and trial")
relevance_by_scenario = {}
for rows in normalized.values():
    for row in rows:
        sid, relevant = row["scenario_id"], set(row["relevant_resource_ids"])
        if sid in relevance_by_scenario and relevance_by_scenario[sid] != relevant:
            raise ValueError(f"Relevance changed between releases/trials for {sid}")
        relevance_by_scenario[sid] = relevant
empty_gold = [sid for sid, relevant in relevance_by_scenario.items() if not relevant]
print("Responses: 1000 before, 1000 after")
print("Scenarios without a relevant resource:", len(empty_gold))
print("Empty gold sets produce zero scores and remain reported.")

## 10. Metric policy from the source evaluation module

In [ ]:
metric_policy = {
    "top1": "any relevant resource at rank 1",
    "hit3": "any relevant resource at ranks 1..3",
    "hit5": "any relevant resource at ranks 1..5",
    "mrr": "reciprocal rank of first relevant item among the returned TOP_K resources",
    "map": "mean per-query AP@K using the project denominator min(total relevant, returned count)",
    "top_k": TOP_K,
    "module": "src/hlo/evaluation/metrics.py",
}
display(pd.DataFrame(metric_policy.items(), columns=["metric", "definition"]))

## 11. Calculate each metric from each response

In [ ]:
measured_rows = []
for release, rows in normalized.items():
    for response in rows:
        ranked = response["ranked_resource_ids"]
        relevant = set(response["relevant_resource_ids"])
        measured_rows.append({"release": release, "scenario_id": response["scenario_id"], "trial": response["trial"],
            "seed": response["seed"], "retrieved_count": len(ranked), "relevant_count": len(relevant),
            "top1": hit_at_k(ranked, relevant, 1), "hit3": hit_at_k(ranked, relevant, 3),
            "hit5": hit_at_k(ranked, relevant, 5), "mrr": reciprocal_rank(ranked, relevant),
            "map": average_precision(ranked, relevant)})
measured = pd.DataFrame(measured_rows)
measured.to_csv(OUTPUT / "measured_response_metrics.csv", index=False)
display(measured.head(10))

## 12. Inspect one scenario from input through result

In [ ]:
selected_id = scenarios.iloc[0].scenario_id
display(scenarios[scenarios.scenario_id.eq(selected_id)])
for release in ("before", "after"):
    for row in normalized[release]:
        if row["scenario_id"] == selected_id:
            print(json.dumps(row, indent=2))
display(measured[measured.scenario_id.eq(selected_id)])

## 13. Assemble scenario columns and two trial columns

In [ ]:
details = {}
for release in ("before", "after"):
    table = scenarios.copy()
    release_rows = measured[measured.release.eq(release)]
    for metric in METRICS:
        pivot = release_rows.pivot(index="scenario_id", columns="trial", values=metric)
        for trial in (1, 2):
            table[f"{metric}_trial{trial}"] = table.scenario_id.map(pivot[trial])
        table[metric] = table[[f"{metric}_trial1", f"{metric}_trial2"]].mean(axis=1)
    if table[METRICS].isna().any().any():
        raise ValueError("Missing measured result")
    details[release] = table
    print(release, table.shape)
display(details["before"].head(5))

## 14. Compute overall and category summaries

In [ ]:
summaries = {release: pd.DataFrame({"metric": METRICS,
    "computed_percent": [table[m].mean() * 100 for m in METRICS]}) for release, table in details.items()}
comparison = pd.DataFrame({"metric": METRICS,
    "before_percent": summaries["before"].computed_percent,
    "after_percent": summaries["after"].computed_percent})
comparison["difference_percentage_points"] = comparison.after_percent - comparison.before_percent
comparison.to_csv(OUTPUT / "response_comparison_summary.csv", index=False)
display(comparison)
category_summaries = {}
for release, table in details.items():
    if "scenario_type" in table:
        category_summaries[release] = table.groupby("scenario_type")[METRICS].mean().mul(100).reset_index()
        display(category_summaries[release])

## 15. Verify supplied ontology fingerprints when present

In [ ]:
def file_hash(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

fingerprint_checks = []
for release, path in ONTOLOGIES.items():
    supplied = set(provenance.loc[provenance.release.eq(release), "ontology_sha256"]) - {""}
    actual = file_hash(path) if path.is_file() else None
    for value in supplied:
        if actual != value:
            raise ValueError(f"Ontology hash mismatch: {release}")
    fingerprint_checks.append({"release": release, "actual_ontology_sha256": actual,
        "response_ontology_hash_present": bool(supplied),
        "ontology_verified_against_response": bool(supplied) and supplied == {actual}})
display(pd.DataFrame(fingerprint_checks))

## 16. Export Before and After Excel workbooks

In [ ]:
from openpyxl.styles import Alignment, Font, PatternFill
workbook_paths = {}
for release, table in details.items():
    destination = OUTPUT / f"HLO_500_{release}_overall_Details.xlsx"
    trial_provenance = provenance[provenance.release.eq(release)].copy()
    with pd.ExcelWriter(destination, engine="openpyxl") as writer:
        table.to_excel(writer, sheet_name="scenarios", index=False)
        summaries[release].to_excel(writer, sheet_name="Summary", index=False)
        trial_provenance.to_excel(writer, sheet_name="Trial_provenance", index=False)
        if release in category_summaries:
            category_summaries[release].to_excel(writer, sheet_name="By_type", index=False)
        for sheet in writer.book.worksheets:
            sheet.freeze_panes = "A2"
            sheet.auto_filter.ref = sheet.dimensions
            sheet.row_dimensions[1].height = 30
            for cell in sheet[1]:
                cell.font = Font(bold=True, color="FFFFFF")
                cell.fill = PatternFill("solid", fgColor="17365D")
                cell.alignment = Alignment(wrap_text=True, vertical="center")
            for column in sheet.columns:
                sheet.column_dimensions[column[0].column_letter].width = min(max(len(str(column[0].value or "")) + 2, 16), 45)
    workbook_paths[release] = destination
    print(destination)

## 17. Verify Excel round trips

In [ ]:
for release, destination in workbook_paths.items():
    actual = pd.read_excel(destination, sheet_name="scenarios", keep_default_na=False)
    actual["scenario_id"] = actual.scenario_id.astype(str)
    pd.testing.assert_frame_equal(actual, details[release], check_dtype=False, atol=1e-12, rtol=0)
    actual_summary = pd.read_excel(destination, sheet_name="Summary")
    pd.testing.assert_frame_equal(actual_summary, summaries[release], check_dtype=False, atol=1e-12, rtol=0)
print("Both 500-row Excel workbooks verified against measured response metrics")

## 18. Compare with historical Excel files without changing targets

In [ ]:
historical_checks = []
for release, new_table in details.items():
    saved_path = EVALUATION / f"results/HLO_500_{release}_overall_Details.xlsx"
    if not saved_path.is_file():
        continue
    saved = pd.read_excel(saved_path, sheet_name=0, keep_default_na=False)
    saved["scenario_id"] = saved.scenario_id.astype(str)
    if set(saved.scenario_id) != SCENARIO_IDS:
        historical_checks.append({"release": release, "metric": "all", "trial": "all", "status": "different_scenario_set", "mismatched_rows": None})
        continue
    saved = saved.set_index("scenario_id")
    actual = new_table.set_index("scenario_id").reindex(saved.index)
    for metric in METRICS:
        for trial in (1, 2):
            column = f"{metric}_trial{trial}"
            if column not in saved:
                continue
            mismatch = ~np.isclose(pd.to_numeric(saved[column]), actual[column], atol=1e-12, rtol=0)
            historical_checks.append({"release": release, "metric": metric, "trial": trial,
                "status": "match" if not mismatch.any() else "different", "mismatched_rows": int(mismatch.sum())})
historical_comparison = pd.DataFrame(historical_checks)
historical_comparison.to_csv(OUTPUT / "historical_excel_comparison.csv", index=False)
display(historical_comparison)

## 19. Preserve normalized raw responses and execution manifest

In [ ]:
for release, rows in normalized.items():
    with (OUTPUT / f"normalized_{release}_responses.jsonl").open("w", encoding="utf-8") as stream:
        for row in rows:
            stream.write(json.dumps(row, ensure_ascii=False) + "\n")
inputs = {"scenarios": SCENARIOS_PATH, "before_response": RESPONSES["before"], "after_response": RESPONSES["after"],
          "metrics_module": PROJECT / "src/hlo/evaluation/metrics.py"}
if RELEVANCE_PATH:
    inputs["relevance"] = RELEVANCE_PATH
manifest = {
    "operation": "calculate trial metrics from scenario-linked raw resource-ranking responses",
    "scenario_count": len(scenarios), "response_count": len(measured), "top_k": TOP_K,
    "metric_policy": metric_policy,
    "input_files": {role: {"path": str(path), "sha256": file_hash(path)} for role, path in inputs.items()},
    "trial_seed_values": {release: sorted(set(provenance.loc[provenance.release.eq(release), "seed"].dropna().astype(int))) for release in ("before", "after")},
    "missing_seed_records": int(provenance.seed.isna().sum()),
    "ontology_fingerprint_checks": fingerprint_checks,
    "historical_execution_provenance_verified": False,
    "historical_excel_comparison": historical_checks,
    "python": platform.python_version(),
    "workbooks": {release: {"path": str(path), "sha256": file_hash(path)} for release, path in workbook_paths.items()},
}
(OUTPUT / "response_evaluation_manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print("Before:", workbook_paths["before"].name)
print("After:", workbook_paths["after"].name)
print("Output folder:", OUTPUT)